In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.tree import DecisionTreeRegressor
# from statsmodels.api import OLS
# from statsmodels.api import add_constant
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

from sklearn.metrics import mean_squared_error

In [2]:
import sklearn
np.__version__, pd.__version__, sklearn.__version__

('2.4.3', '3.0.1', '1.8.0')

### 전처리

In [ ]:
df = pd.read_csv("set_05_data.csv")
df.head(2)

In [4]:
df["minimum_payments"].isna().sum()

np.int64(74)

In [5]:
df["minimum_payments"] = df["minimum_payments"].fillna(df["minimum_payments"].mean())

In [ ]:
df.isna().sum()

In [7]:
df_base = df.copy()
len(df_base)

1000

### Q1. 

In [ ]:
df_q1 = df_base.loc[df_base["purchases"] != 0, ["tenure", "balance", "credit_limit"]].reset_index(drop = True)
len(df_q1)

In [ ]:
df_q1.head(4)

In [ ]:
df_q1["tenure"].unique()

In [ ]:
df_q1_corr_sub = df_q1.loc[df_q1["tenure"] == 6, ["balance", "credit_limit"]].corr()
df_q1_corr_sub

In [ ]:
df_q1_corr = df_q1.groupby("tenure")[["balance", "credit_limit"]].corr().reset_index()
df_q1_corr.head(1)

In [17]:
df_q1_corr_sub = df_q1_corr.loc[df_q1_corr["level_1"] == "balance", ]
df_q1_corr_sub.head(2)

,tenure,level_1,balance,credit_limit
0,6.0,balance,1.0,0.839042
2,7.0,balance,1.0,0.974225


In [19]:
df_q1_corr_sub["credit_limit"].round(2).max()

np.float64(0.97)

### Q2.

In [ ]:
# 단계 1)
df_q2 = df_base.drop(columns = "cust_id")
df_q2.head(1)

In [ ]:
arr_q2_nor = StandardScaler().fit_transform(df_q2)
arr_q2_nor[:1, ]

In [ ]:
# 단계 2)
model_kmeans = KMeans(random_state = 1234)
model_kmeans.fit(arr_q2_nor)

In [26]:
# 단계 3)
ls_k = [2, 3, 4, 5]
k = ls_k[0]
model_kmeans = KMeans(n_clusters = k, random_state = 1234)
model_kmeans.fit(arr_q2_nor)
silhouette_score(arr_q2_nor, labels = model_kmeans.labels_)

0.21544722357637516

https://datadoctorblog.com/2023/09/18/Py-Preprocessing-Binding/

In [27]:
ls_k = [2, 3, 4, 5]
ls_sil = []
for k in ls_k:
    model_kmeans = KMeans(n_clusters = k, random_state = 1234)
    model_kmeans.fit(arr_q2_nor)
    val_sil = silhouette_score(arr_q2_nor, labels = model_kmeans.labels_)
    ls_sil = ls_sil + [val_sil]

In [31]:
ser_sil = pd.Series(ls_sil, index = ls_k) # 굳이 코드를 작성하자면~~
k_best = ser_sil.idxmax()
k_best

np.int64(2)

In [ ]:
# 단계 5)
model_best = KMeans(n_clusters = k_best, random_state = 1234)
model_best.fit(arr_q2_nor)

In [34]:
# 단계 6)
# model_best.labels_
df_q2["cluster"] = model_best.labels_
df_q2.head(2)

,balance,balance_freq,purchases,oneoff_p,installments_p,cash_advance,p_freq,oneoff_p_freq,p_installments_freq,cash_advance_freq,cash_advance_trx,p_trx,credit_limit,payments,minimum_payments,prc_full_payment,tenure,cluster
0,40.900749,0.818182,95.4,0.0,95.4,0.000000,0.166667,0.0,0.083333,0.00,0.0,2.0,1000.0,201.802084,139.509787,0.000000,12.0,0
1,3202.467416,0.909091,0.0,0.0,0.0,6442.945483,0.000000,0.0,0.000000,0.25,4.0,0.0,7000.0,4103.032597,1072.340217,0.222222,12.0,0


In [37]:
df_q2.groupby("cluster")["oneoff_p"].mean().round(2).max()

np.float64(2156.47)

### Q3. 

In [38]:
# 비교, 데이터세트 A, B. A -> 의사결정나무, B -> 나이브베이즈 등등
# 예측값을 concat 한 후 종속변수 평가지표 산출

In [40]:
# 단계 1)
df_q3 = df_base.loc[:, df_base.columns.str.contains("oneoff_p|freq")].copy()
df_q3.head(2)

,balance_freq,oneoff_p,p_freq,oneoff_p_freq,p_installments_freq,cash_advance_freq
0,0.818182,0.0,0.166667,0.0,0.083333,0.00
1,0.909091,0.0,0.000000,0.0,0.000000,0.25


In [42]:
# 단계 2)
df_train = df_q3.loc[df_base["cust_id"] % 4 != 0,  ]
df_test  = df_q3.loc[df_base["cust_id"] % 4 == 0,  ]
len(df_q3), len(df_train), len(df_test)

(1000, 752, 248)

In [44]:
# 단계 3)
df_train_X = df_train.drop(columns = "oneoff_p")
ser_train_y = df_train["oneoff_p"] # 굳이?

In [ ]:
# 단계 4)
model_dt = DecisionTreeRegressor(max_depth = 5, random_state = 1234)
model_dt.fit(X = df_train.drop(columns = "oneoff_p"),
             y = df_train["oneoff_p"])
pred_dt = model_dt.predict(df_test.drop(columns = "oneoff_p"))
pred_dt[:4]

In [57]:
val_R_dt = mean_squared_error(y_true = df_test["oneoff_p"], y_pred = pred_dt) ** 0.5
val_R_dt

3042.987475423554

In [ ]:
# 단계 5)
df_train2 = add_constant(df_train)
df_test2  = add_constant(df_test)
df_test2.head(1)

In [ ]:
model_lr = OLS(exog = df_train2.drop(columns = "oneoff_p"),
               endog = df_train2["oneoff_p"]).fit() # 🤔
pred_lr = model_lr.predict(df_test2.drop(columns = "oneoff_p"))
pred_lr[:4]

In [ ]:
pred_lr2 = np.where(pred_lr < 0, 0, pred_lr)
pred_lr2[:4]

In [56]:
val_R_lr = mean_squared_error(y_true = df_test2["oneoff_p"], y_pred = pred_lr2) ** 0.5
val_R_lr

2126.138317305272

In [58]:
# 단계 6)
round(abs(val_R_dt - val_R_lr), 2)

916.85